# Apex LLM - Fine-Tuning Phi-4 rank=32 | V4 dataset
**Runtime: T4 GPU** (Runtime > Change runtime type > T4 GPU)
Runs in ~15-25 min on T4. Download the LoRA at the end.

In [ ]:
# Cell 1 - GPU check + clone latest Apex_LLM
!nvidia-smi
import torch
print('torch.cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise SystemExit('ERREUR: Pas de GPU. Runtime > Modifier le type > T4 GPU.')

%cd /content
!rm -rf Apex_LLM
!git clone https://github.com/tovrr/Apex_LLM.git
%cd Apex_LLM
!git log --oneline -3

In [ ]:
# Cell 2 - Install dependencies
!pip install --upgrade pip -q
!pip install transformers peft trl accelerate bitsandbytes \
    datasets huggingface_hub -q
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'CUDA must be available'

In [ ]:
# Cell 2b - (Optional) Configure HuggingFace token
# Run this ONLY if you need to download gated models (e.g., unsloth/phi-4-unsloth-bnb-4bit requires login)
from huggingface_hub import login
import getpass

print("Si unsloth/phi-4-unsloth-bnb-4bit demande une authentification :")
print("1. Génère un token sur https://huggingface.co/settings/tokens")
print("2. Colle-le ci-dessous (il sera masqué)")

# Optionnel : configure le token HF
use_token = input("Avez-vous besoin de configurer un HF token? (y/n) [default: n]: ").strip().lower()
if use_token == "y":
    token = getpass.getpass("HuggingFace token (read access): ")
    login(token=token, add_to_git_credential=True)
    print("✅ Token HF configuré")
else:
    print("⏭️  HF token skipped")

In [ ]:
# Cell 3 - Verify training dataset (V4)
import json, os

DATASET_FILE = os.environ.get('APEX_DATASET_FILE', 'dataset_expert_v4.json')
os.environ['APEX_DATASET_FILE'] = DATASET_FILE

with open(DATASET_FILE, encoding='utf-8') as f:
    ds = json.load(f)

print(f'Dataset file: {DATASET_FILE}')
print(f'Dataset size: {len(ds)} examples')
if not ds:
    raise ValueError('Dataset is empty. Upload dataset_expert_v4.json to Colab first.')

print('Sample:', ds[0]['instruction'][:80], '...')
print(f"LoRA config: rank=32, alpha=32, dataset={DATASET_FILE}")
print('Base model: unsloth/phi-4-unsloth-bnb-4bit')

In [ ]:
# Cell 4 - Fine-tuning Phi-4 with LoRA (8-bit + formatting_func)
import json, os, torch
from typing import Any, cast
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

DATASET_FILE = os.environ.get("APEX_DATASET_FILE", "dataset_expert_v4.json")

# ── Formatting function ───────────────────────────────────────────────────────
def format_phi4(examples):
    return [
        f"<|user|>\n{instr}<|end|>\n<|assistant|>\n{out}<|end|>"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]

# ── Model config ──────────────────────────────────────────────────────────────
nom_modele = os.getenv("APEX_BASE_MODEL", "unsloth/phi-4-unsloth-bnb-4bit")
lora_rank = int(os.getenv("APEX_LORA_RANK", "16"))

# ── 8-bit quantization with CPU offload for T4 ────────────────────────────────
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
    llm_int8_compute_dtype=torch.float16,
)

print(f"📥 Loading {nom_modele} (8-bit + CPU offload for T4)...")
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = min(tokenizer.model_max_length, 1024)

modele_base = AutoModelForCausalLM.from_pretrained(
    nom_modele,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation=os.getenv("APEX_ATTENTION_IMPL", "sdpa"),
)
modele_base.config.use_cache = False
modele_base.gradient_checkpointing_enable()

# ── Detect LoRA target modules ────────────────────────────────────────────────
def _detect_target_modules(model: Any) -> list:
    names = [n for n, _ in model.named_modules()]
    if any(n.endswith("qkv_proj") for n in names):
        return ["qkv_proj", "o_proj"]
    if any(n.endswith("q_proj") for n in names) and any(n.endswith("v_proj") for n in names):
        return ["q_proj", "v_proj"]
    raise ValueError("Cannot detect LoRA target modules.")

target_modules = _detect_target_modules(modele_base)
print(f"🔧 LoRA target modules: {target_modules}")

lora_cfg = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_rank,
    target_modules=target_modules,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
modele_apex = get_peft_model(modele_base, lora_cfg)
modele_apex.print_trainable_parameters()

# ── Load dataset ──────────────────────────────────────────────────────────────
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"✅ Dataset loaded: {len(dataset)} examples ({DATASET_FILE})")

# ── Training config (SFTConfig only takes TrainingArguments params) ────────────
training_args = SFTConfig(
    output_dir="./apex_lora_sauvegarde",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    fp16=False,
    bf16=False,
    report_to="none",
)

# ── Monkey-patch: fix Unsloth AttributeError on push_to_hub_token ────────────
if not hasattr(training_args, "push_to_hub_token"):
    training_args.push_to_hub_token = None

# ── Build SFTTrainer with version-adaptive params ────────────────────────────
import inspect
_sft_kwargs = dict(
    model=cast(Any, modele_apex),
    train_dataset=dataset,
    args=training_args,
    formatting_func=format_phi4,
    tokenizer=tokenizer,
)

# Add optional SFTTrainer params if supported by installed trl version
_sft_sig = inspect.signature(SFTTrainer.__init__).parameters
if "max_seq_length" in _sft_sig:
    _sft_kwargs["max_seq_length"] = 1024
if "packing" in _sft_sig:
    _sft_kwargs["packing"] = False

entraineur = SFTTrainer(**_sft_kwargs)

# ── Train ─────────────────────────────────────────────────────────────────────
print("\n⏳ Starting LoRA training (100 steps)...")
entraineur.train()

# ── Save adapter ──────────────────────────────────────────────────────────────
print("\n✅ Training complete!")
cast(Any, modele_apex).save_pretrained("apex_lora_final")
cast(Any, modele_apex).save_pretrained("apex_lora_sauvegarde")
print("💾 Adapter saved to apex_lora_final/ and apex_lora_sauvegarde/")

In [ ]:
# Cell 5 - Download LoRA adapter
import os, shutil
from google.colab import files

for folder in ['apex_lora_final', 'apex_lora_sauvegarde']:
    if os.path.isdir(folder):
        print(f'Found: {folder}')
        !ls -lah $folder/
        shutil.make_archive(folder, 'zip', folder)
        files.download(f'{folder}.zip')
        break
else:
    print('ERROR: No LoRA folder. Last 30 lines of log:')
    !tail -30 finetune.log